# 06 · Complete pipeline: batch, real-time latency and full benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/claverfred/real_time_swahili_english_interpreter/blob/main/notebooks/06_complete_pipeline.ipynb)

Runs the same evaluations as the CPU baseline in the README, on the GPU, so the two are directly comparable:

| Evaluation | Data | CPU baseline (development PC) |
|---|---|---|
| Speech → English (ASR + MT) | all 487 FLEURS `sw_ke` test recordings, human English from FLORES-200 | WER 0.272, chrF 48.4 |
| Real-time latency, TTS on | 30 FLEURS recordings (from #401) streamed at speaking pace | first audio 3.44 / 5.12 / 5.70 s (p50/p95/p99) |
| Text translation | FLORES-200 devtest, 1012 sentences | BLEU 39.0, chrF 61.6 |
| Robustness (optional) | 60 recordings × 9 audio conditions | clean 0.257 … 5 dB noise 0.732 |

Then a batch interpretation of your own file with every artifact exported (transcript, translation, English
speech, events, manifest) and one ZIP of everything. Full run on a T4: roughly 30-60 minutes.

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is enough; L4/A100/L40S are faster). Optional: add a Hugging Face token as the Colab Secret `HF_TOKEN` (key icon on the left); every default model is public, so it is not required.

Results, converted models and data are kept on Google Drive in `MyDrive/swahili_interpreter/`, so later sessions start faster. Nothing is uploaded anywhere else.

In [ ]:
# @title 1. Setup: clone/update the repository, mount Drive, install, fetch data (safe to re-run)
REPO_URL = "https://github.com/claverfred/real_time_swahili_english_interpreter.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
USE_DRIVE = True  # @param {type:"boolean"}
PROFILE = "colab"  # @param {type:"string"}

import os, subprocess, sys
WORKDIR = "/content" if os.path.isdir("/content") else os.getcwd()
REPO = os.path.join(WORKDIR, REPO_URL.rstrip("/").rsplit("/", 1)[-1].removesuffix(".git"))
if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print(subprocess.run(["git", "log", "-1", "--format=commit %h  %cd  %s"], capture_output=True, text=True).stdout)

from src.platform import colab
info = colab.bootstrap(REPO, use_drive=USE_DRIVE)
print("persistent root:", info["root"], "(Google Drive)" if info["on_drive"] else "")
for n in info["notes"]:
    print("NOTE:", n)
print("Hugging Face token:", "from Colab Secrets" if info["hf_login"] else "none (fine: all default models are public)")

r = subprocess.run([sys.executable, "scripts/setup_colab.py", "--profile", PROFILE, "--no-smoke"])
stale = colab.restart_required()
if stale:
    print("Restart the runtime (Runtime > Restart session), then run this cell again:", stale)
assert r.returncode == 0, "setup reported a failure (see the table above)"

In [ ]:
# @title 2. What to run
SPEECH_N = 487  # @param {type:"integer"}
LATENCY_N = 30  # @param {type:"integer"}
MT_N = 1012  # @param {type:"integer"}
ROBUSTNESS = False  # @param {type:"boolean"}
ROBUSTNESS_N = 60  # @param {type:"integer"}

In [ ]:
# @title 3. Speech → English (ASR WER + speech-to-English BLEU/chrF + oracle MT)
runs = []
r = colab.run_cli(["evaluate", "speech", "--n", str(SPEECH_N), "--profile", PROFILE], prefix="eval-speech")
runs.append({"label": "speech", **r})

In [ ]:
# @title 4. Real-time latency (streamed at speaking pace, TTS on)
r = colab.run_cli(["evaluate", "latency", "--n", str(LATENCY_N), "--offset", "400", "--profile", PROFILE],
                  prefix="eval-latency")
runs.append({"label": "latency", **r})

In [ ]:
# @title 5. Text translation (FLORES-200 devtest)
r = colab.run_cli(["evaluate", "mt", "--n", str(MT_N), "--profile", PROFILE], prefix="eval-mt")
runs.append({"label": "mt", **r})

In [ ]:
# @title 6. (Optional) robustness: 9 audio conditions
if ROBUSTNESS:
    r = colab.run_cli(["evaluate", "speech", "--n", str(ROBUSTNESS_N), "--no-oracle", "--bootstrap", "500",
                       "--profile", PROFILE, "--conditions", "clean", "noise_20db", "noise_10db", "noise_5db",
                       "quiet_-20db", "reverb_0.6s", "fast_1.15x", "slow_0.85x", "telephone"], prefix="eval-speech")
    runs.append({"label": "robustness", **r})

In [ ]:
# @title 7. Summary next to the CPU baseline
import pandas as pd
from IPython.display import display
from src.eval.summary import summarize_run

def table(runs, extra_keys=("label", "gpu_peak_mb", "seconds")):
    rows = []
    for r in runs:
        if not r.get("run_dir"):
            rows.append({"label": r.get("label"), "status": f"FAILED (exit {r['returncode']})"})
            continue
        for row in summarize_run(r["run_dir"]):
            rows.append({**{k: r.get(k) for k in extra_keys}, **row, "status": "ok"})
    return pd.DataFrame(rows)

GPU = (colab.environment_report()["cuda"]["gpus"] or [{"name": "cpu"}])[0]["name"].replace(" ", "_")

df = table(runs)
display(df.dropna(axis=1, how="all"))
csv = colab.results_dir() / f"colab_complete_{GPU}.csv"
df.to_csv(csv, index=False)
cpu = pd.DataFrame([
    {"measure": "Swahili WER (487)", "cpu_baseline": 0.272},
    {"measure": "speech→English chrF (487)", "cpu_baseline": 48.38},
    {"measure": "first audio p50 s", "cpu_baseline": 3.44}, {"measure": "first audio p95 s", "cpu_baseline": 5.12},
    {"measure": "first audio p99 s", "cpu_baseline": 5.70},
    {"measure": "FLORES BLEU", "cpu_baseline": 39.03}, {"measure": "FLORES chrF", "cpu_baseline": 61.59}])
def pick(label, col):
    sub = df[df.label == label]
    if "condition" in sub:
        sub = sub[sub.condition.isna() | (sub.condition == "clean")]
    return sub[col].iloc[0] if col in sub and len(sub) else None
cpu["gpu"] = [pick("speech", "wer"), pick("speech", "s2t_chrf"), pick("latency", "first_audio_p50_s"),
              pick("latency", "first_audio_p95_s"), pick("latency", "first_audio_p99_s"), pick("mt", "bleu"), pick("mt", "chrf")]
display(cpu)
print("Note: the GPU run uses the colab profile (MT beam 4, ASR partials on); the CPU baseline used MT beam 2.")

In [ ]:
# @title 8. Batch: interpret your own file with every artifact exported
UPLOAD = False  # @param {type:"boolean"}
INPUT = "data/fleurs_sw/test/" + sorted(os.listdir("data/fleurs_sw/test"))[0]  # default: a FLEURS recording
if UPLOAD:
    INPUT = str(colab.upload_audio(os.path.join(info["root"], "inputs"))[0])
r = colab.run_cli(["interpret", "--input", INPUT, "--export", "--quiet", "--profile", PROFILE], prefix="interpret")
runs.append({"label": "interpret", **r})
from IPython.display import Audio, display
if r["run_dir"]:
    from pathlib import Path
    for f in ("transcript_sw.txt", "translation_en.txt"):
        p = Path(r["run_dir"]) / f
        if p.exists():
            print(f, "->", p.read_text(encoding="utf-8").strip()[:500])
    wav = Path(r["run_dir"]) / "speech_en.wav"
    if wav.exists():
        display(Audio(str(wav)))

In [ ]:
RUN_DIRS, EXTRA_FILES, EXPORT_NAME = [r["run_dir"] for r in runs], [csv], "complete-pipeline"
# @title Export: ZIP this notebook's runs (JSON, CSV, audio, logs, images, manifests) and download
import time
zip_path = colab.results_dir() / f"{EXPORT_NAME}-{time.strftime('%Y%m%d-%H%M%S')}.zip"
env_files = sorted((colab.results_dir().parent / "env").glob("*"))[-4:]
z = colab.zip_runs([p for p in RUN_DIRS if p], zip_path, extra_files=EXTRA_FILES + env_files)
print(z)
colab.download(zip_path)